# Same Algorithm, Forty Different Answers

**Topic:** k-means initialization &nbsp;·&nbsp; **Difficulty:** Medium

Lloyd's algorithm converges. It does not converge to the *right* answer. It
converges to whichever local minimum the initial centroids happen to sit in, and
with random initialization a meaningful fraction of runs land somewhere bad.

Run it 40 times from random starts, look at the spread of final inertias, then
implement k-means++ and watch the spread collapse.

## The data (created in the next cell)

- `X` `(900, 2)`: six well-separated Gaussian blobs of unequal size.
- `k = 6`.
- Use a fresh `np.random.default_rng(2000 + trial)` for each of the two runs in a
  trial, so the random run and the k-means++ run of trial `trial` both start
  from seed `2000 + trial`.

## Your task

With **NumPy only** (no `sklearn.cluster`):

1. Implement Lloyd's algorithm: assign each point to its nearest centroid, move
   each centroid to the mean of its members, repeat 60 times. Inertia is the sum
   of squared distances from points to their assigned centroid.
2. `inertia_random`: run 40 trials initializing centroids as `k` points drawn
   uniformly at random **from the data** (`rng.choice(len(X), k, replace=False)`).
3. `inertia_pp`: run 40 trials with **k-means++** initialization, which picks the first
   centre uniformly, then each subsequent centre with probability proportional to
   its squared distance to the nearest chosen centre.
4. Set `answers` to a dict with `"inertia_random"`, `"inertia_pp"` (length-40
   arrays), `"best_random"`, `"best_pp"`, `"std_random"`, `"std_pp"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks k-means++ has the smaller spread.

In [ ]:
import numpy as np

rng = np.random.default_rng(1616)
centres = np.array([[0, 0], [6, 0], [0, 6], [6, 6], [3, 12], [12, 3]], dtype=float)
sizes = [80, 250, 120, 200, 90, 160]
X = np.vstack([c + rng.normal(0, 0.7, (n, 2)) for c, n in zip(centres, sizes)])
k = 6
print(f"X {X.shape}, k = {k}, cluster sizes {sizes}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("inertia_random", [40], 103438.444892844, 5740.5961262558985, 1e-06),
    ("inertia_pp", [40], 63048.33980223974, -2863.1168549615422, 1e-06),
    ("best_random", [], 944.1235517799512, -158.08217324964903, 1e-06),
    ("best_pp", [], 944.1235517799512, -158.08217324964903, 1e-06),
    ("std_random", [], 1247.7271914637568, -208.91696396876193, 1e-06),
    ("std_pp", [], 984.0707461483686, -164.77085217210578, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "std_pp" in answers and "std_random" in answers:
    _s = answers["std_pp"] < answers["std_random"]
    print(f"  {'k-means++ more stable':<24} {'ok' if _s else 'wrong'}  "
          f"(std {answers['std_pp']:.2f} vs {answers['std_random']:.2f})")
    _ok = _ok and _s

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet, revisit your solution and try again.")

## Solution

Random initialization can put two centres inside one blob and none in another.
Lloyd's algorithm has no way to move a centre across empty space, so that
configuration is a stable local minimum with visibly worse inertia. k-means++
spreads the initial centres by sampling proportional to squared distance, which
makes the bad configurations exponentially unlikely, and it comes with an
`O(log k)` approximation guarantee, which plain random does not.

```python
import numpy as np

def lloyd(C0):
    C = C0.copy()
    for _ in range(60):
        D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
        lab = D.argmin(axis=1)
        for j in range(k):
            m = lab == j
            if m.any():
                C[j] = X[m].mean(axis=0)
    D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
    return float(D.min(axis=1).sum())

def kpp(rng):
    idx = [int(rng.integers(len(X)))]
    for _ in range(k - 1):
        C = X[idx]
        d2 = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2).min(axis=1)
        p = d2 / d2.sum()
        idx.append(int(rng.choice(len(X), p=p)))
    return X[idx]

inertia_random, inertia_pp = [], []
for t in range(40):
    r1 = np.random.default_rng(2000 + t)
    inertia_random.append(lloyd(X[r1.choice(len(X), k, replace=False)]))
    r2 = np.random.default_rng(2000 + t)
    inertia_pp.append(lloyd(kpp(r2)))

inertia_random = np.array(inertia_random)
inertia_pp = np.array(inertia_pp)

answers = {
    "inertia_random": inertia_random,
    "inertia_pp": inertia_pp,
    "best_random": float(inertia_random.min()),
    "best_pp": float(inertia_pp.min()),
    "std_random": float(inertia_random.std()),
    "std_pp": float(inertia_pp.std()),
}
```